In [1]:

import os
print(os.getcwd())
import sys
sys.path.append("QuAlgorithms")
sys.path.append(".")
print(sys.path)

from state_preparation import matrix_msob2
from qsiht_library import generate_binary_values
from qsiht_library import generate_and_shift_array
from qsiht_library import generalFastPath
import qiskit 
from qiskit import QuantumCircuit
import numpy as np


c:\Users\mando\Downloads\QuAlgorithms\QuAlgorithms-master\qu_algorithms
['c:\\Users\\mando\\miniconda3\\envs\\qiskit\\python314.zip', 'c:\\Users\\mando\\miniconda3\\envs\\qiskit\\DLLs', 'c:\\Users\\mando\\miniconda3\\envs\\qiskit\\Lib', 'c:\\Users\\mando\\miniconda3\\envs\\qiskit', '', 'c:\\Users\\mando\\miniconda3\\envs\\qiskit\\Lib\\site-packages', 'c:\\Users\\mando\\miniconda3\\envs\\qiskit\\Lib\\site-packages\\win32', 'c:\\Users\\mando\\miniconda3\\envs\\qiskit\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\mando\\miniconda3\\envs\\qiskit\\Lib\\site-packages\\Pythonwin', 'QuAlgorithms', '.']


In [10]:
import qiskit
from qiskit import QuantumCircuit
import numpy as np
from qiskit.visualization import plot_histogram
import qsiht_library
#np.random.seed(0)
img3 = np.random.rand(2, 2)

f = img3.flatten().astype(float)   # length 64
f = f / np.linalg.norm(f)          # normalize

A , U  = matrix_msob2(f.copy())
print("angles U:", U[1:])
print("A @ f =", A @ f)
        # should have nonzero in first element, ~0 elsewhere

for i, val in enumerate(U):
    print(i, val)




angles U: [-0.80584101 -1.16147591 -0.7416715 ]
A @ f = [ 1.00000000e+00 -5.55111512e-17  2.77555756e-17  0.00000000e+00]
0 1.0
1 -0.8058410103373563
2 -1.161475907335316
3 -0.7416714959769739


In [ ]:
# =========================
# BENCHMARK A (FULL CELL)
# Compare 3 ways to prepare the same 6-qubit (8x8) image state,
# then apply 2D QFT (QFT3 on rows + QFT3 on cols),
# and report transpiled depth + CX/U3 counts.
# =========================

import numpy as np
import sys
sys.path.append("QuAlgorithms-master")

from state_preparation import matrix_msob2

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFT, UnitaryGate, StatePreparation
from qiskit.quantum_info import Statevector

def summarize(name, circuit, basis=("cx","u3"), opt=1):
    """Transpile and return depth + gate counts."""
    tc = transpile(circuit, basis_gates=list(basis), optimization_level=opt)
    ops = tc.count_ops()
    return {
        "name": name,
        "depth": tc.depth(),
        "cx": int(ops.get("cx", 0)),
        "u3": int(ops.get("u3", 0)),
        "ops": dict(ops),
    }

# -------------------------
# 0) Target state: 6 qubits (8x8 random image)
# -------------------------
n = 6
np.random.seed(0)
img = np.random.rand(8, 8)

f = img.flatten().astype(float)      # length 64
f = f / np.linalg.norm(f)            # normalize

# 2D QFT blocks (3 + 3)
#row = [0, 1, 2]
#col = [3, 4, 5]
#qft3 = QFT(3, do_swaps=False).to_gate(label="QFT3")

# -------------------------
# 1) Dense DsiHT-based prep via UnitaryGate(H.T)
# -------------------------
H,U = matrix_msob2(f.copy())           # IMPORTANT: copy() since function edits input
U_prep =UnitaryGate(H.T)                         # prepares |f> from |0...0>

qc_dense = QuantumCircuit(n, name="dense_DsiHT")
qc_dense.append(U_prep, range(n))
#qc_dense.append(qft3, row)
#qc_dense.append(qft3, col)

# sanity check: prep only fidelity
qc_check = QuantumCircuit(n)
qc_check.append(U_prep, range(n))
sv = Statevector.from_instruction(qc_check).data
print("Dense prep fidelity:", float(abs(np.vdot(sv, f))**2))

# -------------------------
# 2) Qiskit initialize(f)
# -------------------------
qc_init = QuantumCircuit(n, name="initialize")
qc_init.initialize(f, range(n))
#qc_init.append(qft3, row)
#qc_init.append(qft3, col)

# -------------------------
# 3) Qiskit StatePreparation(f) (isometry-based)
# -------------------------
qc_sp = QuantumCircuit(n, name="StatePreparation")
qc_sp.append(StatePreparation(f), range(n))
#qc_sp.append(qft3, row)
#qc_sp.append(qft3, col)

# -------------------------
# Transpile + report (basis = cx + u3)
# -------------------------
results = [
    summarize("Dense UnitaryGate(H.T) + 2D QFT", qc_dense, opt=1),
    summarize("initialize(f) + 2D QFT", qc_init, opt=1),
    summarize("StatePreparation(f) + 2D QFT", qc_sp, opt=1),
]

print("\n=== Benchmark Results (transpiled to cx/u3) ===")
for r in results:
    print("\n" + r["name"])
    print("  depth:", r["depth"])
    print("  cx:", r["cx"], " u3:", r["u3"])
    # If you want full breakdown, uncomment:
    # print("  ops:", r["ops"])""""

Dense prep fidelity: 1.0

=== Benchmark Results (transpiled to cx/u3) ===

Dense UnitaryGate(H.T) + 2D QFT
  depth: 3491
  cx: 1783  u3: 2855

initialize(f) + 2D QFT
  depth: 116
  cx: 57  u3: 63

StatePreparation(f) + 2D QFT
  depth: 115
  cx: 57  u3: 63


In [9]:
import numpy 
import qiskit 
import qsiht_library

import qsiht_library
import numpy as np

angles = np.random.randn(63)
circuit = qsiht_library.generalFastPath(6, angles)

decomposed = circuit.decompose()
print(f"After decompose - Total gates: {decomposed.size()}")
print(decomposed.count_ops())

# Or count manually
count = 0
for instruction in circuit.data:
    count += 1
    print(f"Instruction {count}: {instruction.operation.name}")

After decompose - Total gates: 63
OrderedDict({'c5ry_o0': 6, 'c5ry_o1': 5, 'c5ry_o3': 4, 'c5ry_o2': 4, 'c5ry_o7': 3, 'c5ry_o6': 3, 'c5ry_o5': 3, 'c5ry_o4': 3, 'c5ry_o15': 2, 'c5ry_o14': 2, 'c5ry_o13': 2, 'c5ry_o12': 2, 'c5ry_o11': 2, 'c5ry_o10': 2, 'c5ry_o9': 2, 'c5ry_o8': 2, 'c5ry': 1, 'c5ry_o30': 1, 'c5ry_o29': 1, 'c5ry_o28': 1, 'c5ry_o27': 1, 'c5ry_o26': 1, 'c5ry_o25': 1, 'c5ry_o24': 1, 'c5ry_o23': 1, 'c5ry_o22': 1, 'c5ry_o21': 1, 'c5ry_o20': 1, 'c5ry_o19': 1, 'c5ry_o18': 1, 'c5ry_o17': 1, 'c5ry_o16': 1})
Instruction 1: circuit-1523_dg
Instruction 2: circuit-1535_dg
Instruction 3: circuit-1555_dg
Instruction 4: circuit-1591_dg
Instruction 5: circuit-1659_dg
Instruction 6: circuit-1791_dg


In [11]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import RYGate

# ============================================================
# 1) 8x8 IMAGE -> 6-QUBIT (64 AMP) REAL STATEVECTOR
# ============================================================

def generate_random_8x8_image(seed=None):
    if seed is not None:
        np.random.seed(seed)
    return np.random.randint(0, 256, size=(8, 8), dtype=np.int64)

def image_to_statevector(image_8x8):
    flat = image_8x8.flatten().astype(float)  # 64 entries
    norm = np.linalg.norm(flat)
    if norm == 0:
        raise ValueError("Image is all zeros; cannot normalize.")
    return flat / norm, norm


# ============================================================
# 2) YOUR 6-QUBIT FAST-PATH PATTERN: 63 PAIRS
# Bitstrings are MSB..LSB = b5 b4 b3 b2 b1 b0 (length 6).
# ============================================================

def build_pairs_6q_fast_path():
    pairs = []

    # Layer 1: flip b0 for every prefix b5..b1 (32)
    for p in range(2**5):
        prefix = format(p, "05b")      # b5..b1
        pairs.append((prefix + "0", prefix + "1"))

    # Layer 2: flip b1, only when b0=0, for every prefix b5..b2 (16)
    for q in range(2**4):
        prefix = format(q, "04b")      # b5..b2
        pairs.append((prefix + "00", prefix + "10"))  # b1:0->1, b0 fixed 0

    # Layer 3: flip b2 anchor per (b5,b4,b3) (8)
    for r in range(2**3):
        prefix = format(r, "03b")      # b5..b3
        pairs.append((prefix + "000", prefix + "100"))  # b2:0->1, b1=b0=0

    # Layer 4: flip b3 anchor per (b5,b4) (4)
    for u in range(2**2):
        prefix = format(u, "02b")      # b5..b4
        pairs.append((prefix + "0000", prefix + "1000"))  # b3:0->1, b2=b1=b0=0

    # Layer 5: flip b4 anchor per b5 (2)
    for b5 in ["0", "1"]:
        pairs.append((b5 + "00000", b5 + "10000"))        # b4:0->1, b3..b0=0

    # Layer 6: flip b5 global anchor (1)
    pairs.append(("000000", "100000"))

    if len(pairs) != 63:
        raise RuntimeError(f"Expected 63 pairs, got {len(pairs)}.")
    return pairs


# ============================================================
# 3) ANGLES: REAL GIVENS ROTATION TO ZERO OUT j
# Rotation on (ai, aj):
# new_i = cosθ*ai - sinθ*aj
# new_j = sinθ*ai + cosθ*aj
# Choose θ = atan2(-aj, ai)  (equivalently -atan2(aj, ai))
# ============================================================

def compute_angles_real_givens(state64, pairs, eps=1e-15):
    working = np.array(state64, dtype=float, copy=True)
    angles = []

    for (s, t) in pairs:
        i = int(s, 2)
        j = int(t, 2)
        ai, aj = working[i], working[j]

        if abs(ai) < eps and abs(aj) < eps:
            theta = 0.0
        else:
            theta = np.arctan2(-aj, ai)   # == -atan2(aj, ai)

        angles.append(theta)

        c, sn = np.cos(theta), np.sin(theta)
        working[i] = c * ai - sn * aj
        working[j] = sn * ai + c * aj

    return np.array(angles, dtype=float)


# ============================================================
# 4) IMPLEMENT A TWO-LEVEL ROTATION BETWEEN |s> AND |t>
# using an MC-RY on the differing qubit with controls fixed
# to the shared bits (use X for controls-on-0).
#
# IMPORTANT: Qiskit qubit indices: q[0] is LSB (b0), q[5] is MSB (b5).
# Our strings are MSB..LSB (b5..b0), so map:
#   string pos 0 (b5) -> qubit 5
#   string pos 5 (b0) -> qubit 0
# ============================================================

def pos_to_qubit(pos, n):
    return (n - 1) - pos

def add_two_level_ry(qc: QuantumCircuit, s: str, t: str, theta: float):
    n = qc.num_qubits
    if len(s) != n or len(t) != n:
        raise ValueError("Bitstring length mismatch.")
    diffs = [k for k, (a, b) in enumerate(zip(s, t)) if a != b]
    if len(diffs) != 1:
        raise ValueError(f"Pair must differ in exactly one bit. diffs={diffs}")

    diff_pos = diffs[0]
    target = pos_to_qubit(diff_pos, n)

    controls = []
    xed = []
    # Fix all other bits to s[pos]
    for pos in range(n):
        if pos == diff_pos:
            continue
        q = pos_to_qubit(pos, n)
        controls.append(q)
        want = int(s[pos])  # desired control value in computational basis

        # Controls in Qiskit are "on 1". For want=0, X it before & after.
        if want == 0:
            qc.x(q)
            xed.append(q)

    # NOTE: We use RYGate(2*theta) because Qiskit's RY rotates by angle.
    # Our 2x2 real rotation uses theta in the cos/sin matrix; that matches RY(2*theta).
    mcry = RYGate(2.0 * theta).control(len(controls))
    qc.append(mcry, controls + [target])

    # Undo X gates
    for q in reversed(xed):
        qc.x(q)


# ============================================================
# 5) BUILD FULL CIRCUIT + VERIFY
# ============================================================

def build_qsiht_fastpath_circuit_from_image(image_8x8, seed=None):
    # Build state
    state64, norm = image_to_statevector(image_8x8)

    # Pairs & angles
    pairs = build_pairs_6q_fast_path()
    angles = compute_angles_real_givens(state64, pairs)

    # Build circuit
    qc = QuantumCircuit(6)
    qc.initialize(state64, list(range(6)))   # critical: start from generator

    for (s, t), th in zip(pairs, angles):
        add_two_level_ry(qc, s, t, float(th))

    return qc, state64, norm, pairs, angles


def classical_apply_rotations(state64, pairs, angles):
    """Pure numpy verification of your angle schedule."""
    w = np.array(state64, dtype=float, copy=True)
    for (s, t), th in zip(pairs, angles):
        i = int(s, 2)
        j = int(t, 2)
        ai, aj = w[i], w[j]
        c, sn = np.cos(th), np.sin(th)
        w[i] = c * ai - sn * aj
        w[j] = sn * ai + c * aj
    return w


In [6]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit.providers.fake_provider import GenericBackendV2
import numpy as np

n_qubits = 8  # true 8-qubit system = 256-dim

# Random normalized state to force worst-case prep (≈ full QR cost)
rng = np.random.default_rng(42)
raw = rng.random(2**n_qubits) + 1j * rng.random(2**n_qubits)
state = raw / np.linalg.norm(raw)

qc = QuantumCircuit(n_qubits)
qc.append(StatePreparation(state), range(n_qubits))

backend = GenericBackendV2(num_qubits=n_qubits)
qc_t = transpile(qc, backend=backend,
                 basis_gates=['u3', 'cx'],
                 optimization_level=3)

ops = qc_t.count_ops()
print(f"CX gates : {ops.get('cx', 0)}")
print(f"U3 gates : {ops.get('u3', 0)}")
print(f"Depth    : {qc_t.depth()}")

CX gates : 247
U3 gates : 255
Depth    : 495


c:\Users\mando\miniconda3\envs\qiskit\Lib\site-packages\qiskit\compiler\transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


In [7]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Statevector

n = 6
basis = ["u3", "cx"]

# 3 col qubits + 3 row qubits (your encoding)
col_qubits = [0, 1, 2]
row_qubits = [3, 4, 5]

def bit_reverse_bits(x, m):
    b = format(x, f"0{m}b")
    return int(b[::-1], 2)

def permute_state_rowcol_bitrev(state64):
    """
    Bit-reverse the 3-bit col index and 3-bit row index independently.
    Matches the final SWAP stage of QFT(3) on each register.
    """
    out = np.zeros_like(state64, dtype=complex)
    for idx in range(64):
        col = idx & 0b111
        row = (idx >> 3) & 0b111
        col2 = bit_reverse_bits(col, 3)
        row2 = bit_reverse_bits(row, 3)
        idx2 = (row2 << 3) | col2
        out[idx2] = state64[idx]
    return out

# -------------------------
# A) Standard 2D QFT (with swaps)
# -------------------------
qc_std_2d = QuantumCircuit(n)
qc_std_2d.initialize(state64, range(n))
qc_std_2d.append(QFT(3, do_swaps=True).to_gate(label="QFT_col_swaps"), col_qubits)
qc_std_2d.append(QFT(3, do_swaps=True).to_gate(label="QFT_row_swaps"), row_qubits)

# -------------------------
# B) 2D QFT without swaps
# -------------------------
qc_nosw_2d = QuantumCircuit(n)
qc_nosw_2d.initialize(state64, range(n))
qc_nosw_2d.append(QFT(3, do_swaps=False).to_gate(label="QFT_col_nosw"), col_qubits)
qc_nosw_2d.append(QFT(3, do_swaps=False).to_gate(label="QFT_row_nosw"), row_qubits)

# -------------------------
# Verify: std == bitrev(nosw) (up to global phase)
# -------------------------
sv_std = Statevector.from_instruction(qc_std_2d).data
sv_nosw = Statevector.from_instruction(qc_nosw_2d).data
sv_nosw_perm = permute_state_rowcol_bitrev(sv_nosw)

fid = np.abs(np.vdot(sv_std, sv_nosw_perm))**2
print("Fidelity( std , bitrev(nosw) ) =", fid)

# -------------------------
# Hardware-model cost compare
# -------------------------
t_std = transpile(qc_std_2d, basis_gates=basis, optimization_level=3)
t_nosw = transpile(qc_nosw_2d, basis_gates=basis, optimization_level=3)

def summarize(name, circ):
    ops = circ.count_ops()
    print(f"\n=== {name} ===")
    print("depth:", circ.depth())
    print("cx:", ops.get("cx", 0))
    print("u3:", ops.get("u3", 0))
    print("total ops:", sum(ops.values()))
    print("ops:", ops)

summarize("Standard 2D QFT (with swaps)", t_std)
summarize("2D QFT (no swaps)", t_nosw)

C:\Users\mando\AppData\Local\Temp\ipykernel_22484\1816588184.py:37: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc_std_2d.append(QFT(3, do_swaps=True).to_gate(label="QFT_col_swaps"), col_qubits)
C:\Users\mando\AppData\Local\Temp\ipykernel_22484\1816588184.py:38: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc_std_2d.append(QFT(3, do_swaps=True).to_gate(label="QFT_row_swaps"), row_qubits)
C:\Users\mando\AppData\Local\Temp\ipykernel_22484\1816588184.py:45: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is depre

Fidelity( std , bitrev(nosw) ) = 0.999999999999998

=== Standard 2D QFT (with swaps) ===
depth: 127
cx: 69
u3: 79
total ops: 154
ops: OrderedDict({'u3': 79, 'cx': 69, 'reset': 6})

=== 2D QFT (no swaps) ===
depth: 127
cx: 69
u3: 79
total ops: 154
ops: OrderedDict({'u3': 79, 'cx': 69, 'reset': 6})
